# Independent reasoning after assistance
Use a single study version and the proposed v1 export dictionary. Each scheduled task has a row even when outcomes are missing. This descriptive notebook reports complete-form accuracy for independent assessments and available-case summaries for training. A participant is the unit of resampling. The optional planning section uses explicit assumptions independent of the extract.

In [ ]:
import csv
import math
import os
import random
from collections import defaultdict
from pprint import pprint
from statistics import NormalDist, mean, median

fields = {
    'participant_id', 'condition', 'phase', 'task_id', 'form_sequence',
    'initial_choice', 'final_choice', 'initial_correct', 'final_correct',
    'initial_confidence', 'final_confidence', 'ai_shown', 'ai_choice', 'initial_seconds'
}
conditions = ('conventional', 'metacognitive', 'structured_reflection')
phases = ('baseline', 'training', 'post', 'follow_up')

def number(value, low, high=None, integer=False):
    if value == '':
        return None
    result = int(value) if integer else float(value)
    if not math.isfinite(result) or result < low or (high is not None and result > high):
        raise ValueError('Number outside the allowed range')
    return result

def read_extract(path):
    rows, seen, assignments = [], set(), {}
    with open(path, newline='', encoding='utf-8-sig') as stream:
        reader = csv.DictReader(stream)
        if set(reader.fieldnames or []) != fields or len(reader.fieldnames) != len(fields):
            raise ValueError('Expected exactly the columns listed in the v1 data dictionary')
        for index, row in enumerate(reader, 2):
            try:
                if None in row or None in row.values():
                    raise ValueError('Incorrect number of fields')
                key = tuple(row[k] for k in ('participant_id', 'phase', 'task_id'))
                if not all(key) or key in seen or row['phase'] not in phases:
                    raise ValueError('Missing, repeated or invalid scheduled task')
                seen.add(key)
                assignment = row['condition'], row['form_sequence']
                if assignment[0] not in conditions or not assignment[1]:
                    raise ValueError('Missing or invalid allocation')
                if assignments.setdefault(key[0], assignment) != assignment:
                    raise ValueError('Assignment changed across rows')
                for stage in ('initial', 'final'):
                    for suffix, upper in [('correct', 1), ('confidence', 100)]:
                        name = f'{stage}_{suffix}'
                        row[name] = number(row[name], 0, upper, integer=True)
                        if row[name] is not None and not row[f'{stage}_choice']:
                            raise ValueError('Outcome without an observed choice')
                row['initial_seconds'] = number(row['initial_seconds'], 0)
                if row['ai_shown'] not in ('0', '1'):
                    raise ValueError('Missing or invalid exposure')
                shown = row['ai_shown'] == '1'
                if (row['ai_choice'] and not shown) or (shown and not row['initial_choice']):
                    raise ValueError('Inconsistent AI exposure')
                if row['phase'] != 'training' and (shown or row['final_choice']):
                    raise ValueError('Unaided assessments use only initial decisions')
            except (TypeError, ValueError) as exc:
                raise ValueError(f'Invalid record on line {index}: {exc}') from None
            rows.append(row)
    return rows

In [ ]:
def brier(confidence, correct):
    if confidence is None or correct is None:
        return None
    return (confidence / 100 - correct) ** 2

def trial_metrics(row):
    initial, final, advice = (row[k] for k in ('initial_choice', 'final_choice', 'ai_choice'))
    return {
        'initial_correct': row['initial_correct'],
        'initial_confidence_pct': row['initial_confidence'],
        'initial_brier': brier(row['initial_confidence'], row['initial_correct']),
        'initial_seconds': row['initial_seconds'],
        'final_correct': row['final_correct'],
        'final_confidence_pct': row['final_confidence'],
        'final_brier': brier(row['final_confidence'], row['final_correct']),
        'revised': int(final != initial) if initial and final else None,
        'adopted_on_disagreement': int(final == advice) if initial and final and advice and initial != advice else None,
    }

def percentile(sorted_values, p):
    offset = p * (len(sorted_values) - 1)
    left, right = math.floor(offset), math.ceil(offset)
    return sorted_values[left] + (offset - left) * (sorted_values[right] - sorted_values[left])

def summary(person_values):
    values = [v for v in person_values if v is not None]
    interval = None
    if len(values) >= 2:
        rng = random.Random(7193)
        means = sorted(mean(rng.choices(values, k=len(values))) for _ in range(2000))
        interval = [round(percentile(means, p), 4) for p in (0.025, 0.975)]
    return dict(n=len(values), missing_people=len(person_values)-len(values),
                estimate=round(mean(values), 4) if values else None, bootstrap_95=interval)

def describe(rows):
    by_group = defaultdict(lambda: defaultdict(list))
    for row in rows:
        by_group[row['condition'], row['phase']][row['participant_id']].append(trial_metrics(row))
    for (arm, phase), people in sorted(by_group.items()):
        print(arm, phase, 'scheduled participants:', len(people))
        for metric in next(iter(people.values()))[0]:
            if phase != 'training' and metric.startswith(('final', 'revised', 'adopted')):
                continue
            person_values = []
            observed = scheduled = 0
            for trials in people.values():
                values = [trial[metric] for trial in trials if trial[metric] is not None]
                observed += len(values)
                scheduled += len(trials)
                complete_required = phase != 'training' and metric == 'initial_correct'
                if not values or (complete_required and len(values) != len(trials)):
                    person_values.append(None)
                else:
                    person_values.append(median(values) if metric == 'initial_seconds' else mean(values))
            print(metric, dict(observed_trials=observed, scheduled_trials=scheduled, **summary(person_values)))

# Arithmetic checks use constructed inputs. They are never passed into study reporting.
assert brier(80, 1) == (0.8 - 1) ** 2
assert brier(100, 0) == 1 and brier(None, 1) is None
example = dict(initial_choice='a', final_choice='b', ai_choice='b', initial_correct=0,
               final_correct=1, initial_confidence=50, final_confidence=80, initial_seconds=30)
assert trial_metrics(example)['adopted_on_disagreement'] == 1
assert trial_metrics({**example, 'initial_choice': 'b'})['adopted_on_disagreement'] is None
assert summary([None, 0.5]) == dict(n=1, missing_people=1, estimate=0.5, bootstrap_95=None)

## Descriptive results
For each arm, phase and outcome the estimate weights participants equally. Training accuracy and confidence use the available responses at each decision stage and can have different denominators. Revision uses paired choices. Adoption requires a displayed, codable recommendation initially disagreed with and an observed final choice. Response time is the mean of within-person medians in seconds. Bootstrap intervals are percentile intervals from 2,000 resamples within each arm and phase; small samples and missingness can make them unreliable. Inapplicable and missing observations are both outside the outcome-specific denominator. No significance tests are run.

In [ ]:
extract_path = os.getenv('THINKTRACE_CSV')
if extract_path:
    rows = read_extract(extract_path)
    print('Scheduled task rows:', len(rows))
    describe(rows)
else:
    print('Set THINKTRACE_CSV to analyze a reviewed extract. No study results loaded.')

## Planning sensitivity
Set a standardized mean difference using the smallest meaningful raw accuracy difference divided by an evidence-based participant standard deviation. Specify type I error, power and attrition. A single pairwise primary comparison is assumed, with equal allocation across three arms. The calculation is a normal approximation and does not account for item structure, baseline adjustment, multiplicity or attrition bias. Simulate the intended design before finalizing enrollment.

In [ ]:
def planned_enrollment(effect_sd, alpha, power, attrition):
    if not (math.isfinite(effect_sd) and effect_sd > 0 and 0 < alpha < 1 and 0.5 < power < 1 and 0 <= attrition < 1):
        raise ValueError('Review planning inputs')
    normal = NormalDist()
    z = normal.inv_cdf(1 - alpha / 2) + normal.inv_cdf(power)
    complete = math.ceil(2 * (z / effect_sd) ** 2)
    assigned = math.ceil(complete / (1 - attrition))
    return dict(complete_per_arm=complete, assign_per_arm=assigned, total_for_three_arms=3 * assigned)

# Supply approved planning assumptions before obtaining a study-specific target.
planning_assumptions = None
if planning_assumptions is not None:
    pprint(planned_enrollment(**planning_assumptions))
assert planned_enrollment(0.5, 0.05, 0.8, 0)['complete_per_arm'] == 63